# 第 02 课 - 探索智能体框架

智能体框架提供了一小组可组合的积木，让你不必自己编写模型调用循环、工具分发代码和对话记录的维护逻辑。在 LangChain 和 LangGraph 里，这些积木是：

- **模型客户端（Model client）**——连接 AI 模型接口并负责通信
- **智能体（Agent）**——把模型客户端、指令和工具定义包在一起，运行工具调用循环
- **工具（Tools）**——用模型可以调用的自定义函数扩展智能体的能力
- **记忆（线程）**——一个检查点器（checkpointer），保存对话历史，让多轮对话得以进行

本课我们会用这些概念构建一个**旅行预订智能体**，用来查询目的地是否可预订。

## 环境准备

前置条件：在仓库根目录运行 `pip install -r requirements.txt`，把 `.env.example` 复制为 `.env`，填入 `LLM_BASE_URL`、`LLM_API_KEY`、`LLM_MODEL`，然后运行 `python scripts/check_endpoint.py`。

下面的单元会从 `.env` 加载这些变量并构建聊天模型客户端。`ChatOpenAI` 使用 OpenAI Chat Completions 协议，所以同一段代码可以对接 DeepSeek、OpenAI、本地 Ollama 服务或任何其他兼容接口——只需要改这三个环境变量。`LLM_EXTRA_BODY` 用来传递可选的、特定于服务商的请求选项（对 DeepSeek 来说，它会关闭思考模式）。

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"缺少必需的环境变量：{'、'.join(missing)}。"
        "请把仓库根目录下的 .env.example 复制为 .env 并填好。"
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"模型客户端已就绪：{os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

模型客户端已就绪：deepseek-v4-pro @ https://api.deepseek.com/v1


## 理解框架架构

这些组件按层次组合在一起：

```
ChatOpenAI  →  create_agent  →  tools
                             →  checkpointer (thread)
```

1. **模型客户端**——`ChatOpenAI` 只负责 HTTP 通信。它按 Chat Completions API 的格式组装请求，发送到 `LLM_BASE_URL`，再解析回复（包括模型请求的任何工具调用）。它对循环和对话一无所知。
2. **智能体**——`create_agent` 掌管**工具调用循环**：把对话发给模型，执行模型请求的工具，把结果追加进去，如此反复，直到模型以纯文本作答。你传入的指令会成为系统提示词。
3. **工具**——用 `@tool` 装饰的普通 Python 函数。智能体把根据函数签名和 docstring 生成的 schema 交给模型，并在模型请求时调用该函数。
4. **记忆（线程）**——**检查点器**在每一步之后保存消息历史。每个对话由一个 `thread_id` 标识；传同一个 id，智能体就记得之前的轮次；传一个新的 id，就从头开始。

下面我们一层一层地构建。模型客户端已经在上面的环境准备单元里创建好了。

## 用 @tool 装饰器添加工具

工具让智能体能做生成文本之外的事。`@tool` 装饰器把一个普通的 Python 函数变成智能体可以调用的东西。

要点：
- **docstring** 会成为模型看到的工具描述。
- **类型注解**定义了参数的 schema。
- `Annotated[type, "description"]` 为每个参数添加描述，让模型明白该传什么。
- 模型请求时工具会自动运行；后面的课程会展示如何先经过人工批准。

In [2]:
from typing import Annotated

from langchain.tools import tool


@tool
def check_destination_availability(
    destination: Annotated[str, "要查询是否可预订的目的地"]
) -> str:
    """查询某个度假目的地当前是否可以预订。"""
    available = {
        "成都": True,
        "西安": True,
        "哈尔滨": False,
        "三亚": True,
        "拉萨": False,
    }
    is_available = available.get(destination, False)
    return f"{destination}{'可以预订' if is_available else '暂不可预订'}。"

## 创建带工具的智能体

现在把模型客户端、指令和工具组合成一个智能体。`system_prompt` 定义了智能体的角色和行为方式。

我们还挂上了一个 `InMemorySaver` **检查点器**。没有它，智能体在两次 `invoke` 调用之间会忘掉一切；有了它，每个由 `thread_id` 标识的对话都会被保存下来，并在下一轮回放。

`agent.invoke` 返回完整的消息历史——用户消息、模型发起的工具调用、工具结果，以及最终回复。`reply_text` 从最后一条消息里取出文本（有些服务商返回的 content 是一组内容块的列表，而不是单个字符串）。

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    llm,
    tools=[check_destination_availability],
    system_prompt=(
        "你是一名旅行预订助手，帮助用户查询目的地是否可预订并给出推荐。"
        "推荐任何目的地之前，一定要先查询它是否可预订。请用中文回答。"
    ),
    checkpointer=InMemorySaver(),
)


def reply_text(result) -> str:
    """返回最后一条消息的文本；content 可能是字符串，也可能是内容块列表。"""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content

## 用线程进行多轮对话

**线程（thread）**是检查点器记住的一段对话。你通过一个配置字典 `{"configurable": {"thread_id": "..."}}` 来选择它，作为第二个参数传给 `invoke`。每次调用都传同一个配置，智能体就能访问完整的对话历史，并回溯之前的消息。

智能体在任何一轮都可以调用 `check_destination_availability`——工具调用循环在每次 `invoke` 内部都会运行。

这个工具只回答你*点名*的目的地；做完本 notebook 之后，添加一个 `list_destinations` 工具是个不错的练习。

In [4]:
session = {"configurable": {"thread_id": "travel-availability-1"}}

# 第 1 轮：询问几个具体的目的地
result = agent.invoke(
    {"messages": [{"role": "user", "content": "成都、西安、哈尔滨、三亚和拉萨，哪些可以预订？"}]},
    session,
)
print(f"智能体：{reply_text(result)}")

# 第 2 轮：追问——智能体记得之前的对话
result = agent.invoke(
    {"messages": [{"role": "user", "content": "我想去暖和一点的地方，有哪些可以订？"}]},
    session,
)
print(f"\n智能体：{reply_text(result)}")

智能体：查询结果如下：

**✅ 可以预订：**
- 成都
- 西安
- 三亚

**❌ 暂不可预订：**
- 哈尔滨
- 拉萨

如果您计划出行，建议优先考虑成都、西安或三亚。需要我为您推荐其中的某个目的地，或了解更详细的信息吗？

智能体：根据您的需求，想要去暖和一点的地方，可以预订并符合的是 **三亚**。

三亚位于海南岛，属于热带海洋性气候，冬季温暖宜人，非常适合想要避寒、享受阳光沙滩的旅客。

需要我为您推荐三亚的行程或更多相关信息吗？


## 第二个工具与流式输出

工具可以拥有自己的状态。下面的工具随机挑选一个度假目的地，并记住上一次的选择，这样连续两次调用永远不会返回同一个地方——当用户否决了一个建议、要求再来一个时，这很有用。

我们围绕它创建第二个智能体，同样带检查点器，这次以**流式**方式接收回复。`agent.astream(..., stream_mode="messages")` 会为图产生的每一个消息块产出一对 `(token, metadata)`；我们打印来自模型节点的块，并在工具节点运行时另起一行，这样工具调用前后的文本就不会连在一起。Jupyter 支持顶层 `await`，所以下面的 `async for` 可以直接运行。

两轮共用一个 `thread_id`，因此在第二轮时智能体知道哪个目的地被否决了。

In [8]:
import random

# 工具可以从中挑选的度假目的地列表。
_DESTINATIONS = [
    "四川成都",
    "陕西西安",
    "黑龙江哈尔滨",
    "海南三亚",
    "西藏拉萨",
    "云南大理",
    "浙江杭州",
    "广西桂林",
    "新疆喀什",
    "福建厦门",
]

# 记住上一次的目的地，避免连续两次调用给出同一个地方。
_last_destination: str | None = None


@tool
def get_random_destination() -> str:
    """随机提供一个度假目的地。"""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


random_trip_agent = create_agent(
    llm,
    tools=[get_random_destination],
    system_prompt="你是一个乐于助人的 AI 智能体，可以为客户随机挑选目的地并规划度假行程。请用中文回答。",
    checkpointer=InMemorySaver(),
)

user_inputs = [
    "帮我规划一个一日游。",
    "我不喜欢这个目的地，换一个地方再帮我规划一次。",
]
session = {"configurable": {"thread_id": "random-trip-1"}}

for turn, user_input in enumerate(user_inputs, start=1):
    print(f"--- 第 {turn} 轮 ---")
    print(f"用户：{user_input}\n智能体：", end="")
    async for token, metadata in random_trip_agent.astream(
        {"messages": [{"role": "user", "content": user_input}]}, session, stream_mode="messages"
    ):
        if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
            print(token.content, end="", flush=True)
        elif metadata.get("langgraph_node") == "tools":
            print()  # 在工具调用前后的文本之间换行
            print("===")
    print("\n")

--- 第 1 轮 ---
用户：帮我规划一个一日游。
智能体：我来帮您规划一日游！首先让我随机挑选一个目的地。
===
恭喜！这次随机挑选到的目的地是 **新疆喀什** 🎉

喀什是一座充满异域风情的历史文化名城，被誉为"不到喀什，就不算到新疆"。以下是为您精心规划的一日游行程：

---

## 🏜️ 喀什一日游攻略

### 🌅 上午：感受古城韵味

**9:00 — 喀什古城开城仪式**
- 在古城东门观看精彩的入城仪式表演，维吾尔族歌舞热情洋溢，是开启一天的最佳方式。

**9:30 — 漫步喀什古城**
- 穿梭在迷宫般的街巷中，欣赏土黄色的生土建筑、雕花木门、彩色窗户。
- 推荐打卡：百年老茶馆、职人巴扎（铁器、木器、铜器手工艺街）。

**11:30 — 艾提尕尔清真寺**
- 新疆最大的清真寺，外观庄严肃穆，广场上鸽子成群，适合拍照和感受宗教文化氛围。

### 🍽️ 中午：地道美食

**13:00 — 午餐时间**
- 推荐品尝：羊肉抓饭、烤包子、拉条子、缸子肉。
- 百年老茶馆里点一壶砖茶，配上一块馕，看当地老人弹唱，别有一番风味。

### 🏛️ 下午：历史与购物

**15:00 — 香妃园**
- 参观传说中的"香妃"家族墓地，了解当地建筑艺术与历史传说。

**17:00 — 喀什大巴扎（中西亚国际贸易市场）**
- 亚洲最大的集市之一，可以淘到干果、地毯、民族乐器、手工艺品等，记得体验砍价的乐趣！

### 🌇 傍晚：美食收尾

**19:30 — 夜市美食之旅**
- 喀什的夜市热闹非凡，烤羊肉串、酸奶粽子、胡辣羊蹄、石榴汁……边逛边吃，结束完美的一天。

---

## 💡 温馨提示
- **时间**：新疆与内地有时差，实际作息约晚 2 小时，行程可灵活调整。
- **穿着**：早晚温差大，建议带一件外套；古城徒步多，穿舒适的鞋子。
- **防晒**：紫外线较强，注意防晒补水。
- **礼仪**：尊重当地民族宗教习俗，进入清真寺等场所注意着装。

---

祝您在喀什度过愉快的一天！如果您希望更换目的地，或者想针对某个环节（如美食、购物、亲子等）做更详细的定制，随时告诉我哦～ 😊

--- 第 2 轮 ---
用户：我不喜欢这个目的地，换一个地方再帮我规划一次。
智能体：好的，没问题！让我重新为您随机挑选一个目的地。
===
这

## 小结

本课你探索了智能体框架的四块积木：

| 概念 | 你学到了什么 |
|---------|------------------|
| **模型客户端** | `ChatOpenAI` 可以对接任何 OpenAI 兼容接口；服务商只是一个配置项 |
| **智能体** | `create_agent` 把模型客户端、指令和工具打包在一起，运行工具调用循环 |
| **工具** | `@tool` 装饰器把 Python 函数（docstring + 类型注解）暴露给智能体调用 |
| **记忆** | `InMemorySaver` 加上 `thread_id`，在多轮对话之间维护对话历史 |

这些积木组合起来，就能创建出能自然对话、调用外部函数并保持上下文的智能体——这是后续课程中更高级智能体模式的基础。